# 0. Setup: montar Drive y clonar repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO_PATH = '/content/drive/MyDrive/tfg-datos'
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/monica793/tfg-datos.git {REPO_PATH}
else:
    !git -C {REPO_PATH} pull
print('Repo listo en', REPO_PATH)

In [ ]:
import sys
os.chdir('/content/drive/MyDrive/tfg-datos')
sys.path.insert(0, '/content/drive/MyDrive/tfg-datos')

# 1. Instalar Sionna (solo si no está)

In [ ]:
try:
    import sionna.phy
    print('Sionna ya instalado')
except ImportError:
    !pip install sionna
    print('Sionna instalado — reinicia el runtime si hay errores')

# 2. Pipeline híbrido — entrenar y evaluar vs R

In [ ]:
from training.train_hybrid import train_ae_for_n, k_is_valid_for_5g, N_FIXED, RHO_DBS, K_CAND

hybrid_models = {}
for rho_db in RHO_DBS:
    valid_ks = [k for k in K_CAND if k < N_FIXED and k_is_valid_for_5g(k, N_FIXED)]
    k_train  = valid_ks[len(valid_ks) // 2]
    hybrid_models[rho_db] = train_ae_for_n(n=N_FIXED, rho_db=rho_db, k_train=k_train)

In [ ]:
# Curvas Pfa/Pmd/Pie vs R para el híbrido
# (tiene sentido porque el Polar se reconstruye para cada k)
from evaluation.plot_pfa_pmd_pie import run_curves_for_n
from systems.hybrid_polar import ActivityAwarePolarSystem

for rho_db, ae in hybrid_models.items():
    def make_hybrid(k, n, _ae=ae):
        return ActivityAwarePolarSystem(k=k, n=n, ae_model=_ae, p_empty=0.3, thresh=0.5)
    run_curves_for_n(n=N_FIXED, rho_db=rho_db, make_system=make_hybrid,
                     label='Híbrido Polar+AE')

# 3. End-to-end — entrenar con k=50 fijo

In [ ]:
from training.train_e2e import train_e2e, K, N, RHO_DBS as E2E_RHO_DBS

e2e_models = {}
for rho_db in E2E_RHO_DBS:
    encoder, decoder = train_e2e(k=K, n=N, rho_db=rho_db)
    e2e_models[rho_db] = (encoder, decoder)

print('Modelos e2e listos:', list(e2e_models.keys()))

# 4. Comparativa híbrido vs end-to-end vs baseline — curvas vs SNR

Aquí fijamos k=50, n=100 y barremos Eb/No.
Esta es la comparativa correcta para el end-to-end.

In [ ]:
from evaluation.plot_pie_vs_snr import run_curves_vs_snr
from systems.hybrid_polar import ActivityAwarePolarSystem
from systems.e2e_system import E2ESystem
from systems.baseline_polar import BaselinePolarSystem

# Usamos los modelos entrenados con rho=0.0 dB
# (el SNR de entrenamiento — puedes cambiar a 3.0)
RHO_TRAIN = 0.0
ae       = hybrid_models[RHO_TRAIN]
enc, dec = e2e_models[RHO_TRAIN]

systems = {
    'Baseline Polar':   lambda: BaselinePolarSystem(k=50, n=100, p_empty=0.3),
    'Híbrido Polar+AE': lambda _ae=ae: ActivityAwarePolarSystem(
                            k=50, n=100, ae_model=_ae, p_empty=0.3, thresh=0.5),
    'End-to-End MLP':   lambda _e=enc, _d=dec: E2ESystem(
                            k=50, n=100, encoder=_e, decoder=_d, p_empty=0.3, thresh=0.5),
}

run_curves_vs_snr(systems, k=50, n=100)

# 5. (Opcional) Repetir comparativa para rho=3.0 dB

In [ ]:
RHO_TRAIN = 3.0
ae       = hybrid_models[RHO_TRAIN]
enc, dec = e2e_models[RHO_TRAIN]

systems = {
    'Baseline Polar':   lambda: BaselinePolarSystem(k=50, n=100, p_empty=0.3),
    'Híbrido Polar+AE': lambda _ae=ae: ActivityAwarePolarSystem(
                            k=50, n=100, ae_model=_ae, p_empty=0.3, thresh=0.5),
    'End-to-End MLP':   lambda _e=enc, _d=dec: E2ESystem(
                            k=50, n=100, encoder=_e, decoder=_d, p_empty=0.3, thresh=0.5),
}

run_curves_vs_snr(systems, k=50, n=100)